# Sequences

Recurrent models (RNN, LSTM, GRU) process sequences where each step's output depends on the
previous ones. They implement the `Nn.Recurrent` interface: `recurStep` consumes the cell
(a linear resource), takes one input vector, and threads back the output plus the updated
cell; `recurReset` clears the hidden state between sequences.

## The recurrent interface

`recurStep` is one timestep; fold it over a sequence to process the whole thing:

In [ ]:
:t recurStep

In [ ]:
:t recurReset

## Building a cell

`rnn`, `lstm`, `gru` are `Init` builders, same as the feedforward layers. Build an RNN cell
(1 input feature → 4 hidden) and realise it with `runInitL`:

In [ ]:
mkRnn : Init (Rnn 1 4 TapeExecutor F64 WithGrad)
mkRnn = rnn ttanh

## One timestep

Feed a single input vector through the cell and read the hidden output:

In [ ]:
:exec run (do {
  cell <- runInitL mkRnn;
  x <- liftIO1 (tensor {dims=[1]} {ex=TapeExecutor} {dt=F64} (Const 1.0));
  (MkBang h # cell1) <- recurStep cell (retypeGrad x);
  discard cell1;
  liftIO1 (putStrLn ("hidden[0] = " ++ show (primItem1d {ex=TapeExecutor} h.tensorPtr 0))) })

## Folding over a sequence

To process a full sequence you fold `recurStep` across its timesteps, threading the cell
through each. For supervised recurrent training, the loss function does exactly that —
folding the per-step losses into one scalar — and is handed to `fit` as a custom
`EpochStep`. The worked examples are `make example-rnn` and `make example-lstm`
(`packages/idris-ml-examples/src/Example/Rnn.idr`).

## LSTM and GRU: drop-in replacements

LSTM handles longer dependencies; GRU is a lighter-gated variant. The interface is identical — swap the builder:

In [ ]:
:t lstm

In [ ]:
:t gru

## When to use what

| Model | Best for | Trade-off |
|-------|----------|-----------|
| RNN | short sequences, simple patterns | fast, but forgets long-range structure |
| LSTM | medium sequences, complex patterns | more parameters, better memory |
| GRU | similar to LSTM, fewer parameters | simpler gating, often comparable |
| Transformer | long sequences, parallel | `make example-transformer` |

For full training runs (the kernel buffers output, so long runs are impractical here):

```bash
make example-rnn      # RNN on a repeating-pattern task
make example-lstm     # LSTM on the same task with early stopping
```

Next: [07 Device safety](07_device_safety.ipynb).